# Model 5 - Review Sentiment Analysis

Roumeliotis, Tselikas & Nasiopoulos (2024). LLMs in e-commerce: A comparative
analysis of GPT and LLaMA models in product review evaluation.
Natural Language Processing Journal (Elsevier), Vol. 6, 100056.
DOI: 10.1016/j.nlp.2024.100056

reads slv_reviews, runs ai_query() with a structured JSON prompt on each review,
extracts sentiment, score and issues, joins to churn predictions, writes gld_review_insights.

no MLflow registration. this is a prompted LLM call, not a trained model.
runs after 09d_batch_scoring - gld_churn_predictions must exist before this notebook runs.

In [0]:
from pyspark.sql.functions import (
    col, expr, get_json_object, when,
    count, avg, round as spark_round,
    current_date, desc, trim
)

print("imports done")

**Read Silver Reviews**

In [0]:
reviews_df = spark.table("bharatmart.silver.slv_reviews")

print(f"total reviews : {reviews_df.count():,}")

In [0]:
display(reviews_df.limit(5))

**Check Coverage**

In [0]:
total = reviews_df.count()
has_text   = reviews_df.filter(
    col("review_text").isNotNull() & (col("review_text") != "null")).count()
ghost = reviews_df.filter(col("_is_ghost_order") == True).count()
no_text  = total - has_text

print(f"total reviews    : {total:,}")
print(f"has review_text  : {has_text:,}  ({has_text/total*100:.1f}%)")
print(f"rating only : {no_text:,}  ({no_text/total*100:.1f}%)")
print(f"ghost orders : {ghost:,}  (will be excluded before scoring)")
print("rating-only reviews are expected — 40% of customers skip the text field")

411,470 reviews in total. 92.5% have actual text, much higher than the 40% estimate
in the design doc. that estimate was wrong.

30,835 rating-only reviews will be skipped. these customers gave a star but wrote nothing,
so there is nothing for ai_query() to analyse.

12,286 ghost orders are in the reviews table. these have no valid customer link so they
cannot join to churn predictions. they will be excluded before scoring, same as every
other model in the pipeline.

380,635 reviews will go through ai_query().

**Filter to Reviews With Text**

In [0]:
text_reviews_df = reviews_df.filter(
    col("review_text").isNotNull() &
    (trim(col("review_text")) != "") &
    (col("review_text") != "null") &
    (col("_is_ghost_order") == False))

before = reviews_df.count()
after = text_reviews_df.count()
print(f"before : {before:,}")
print(f"after : {after:,} (excluded rating-only + ghost orders)")

42,240 reviews dropped. 30,835 had no text, 12,286 were ghost orders.
881 were both, so the actual drop is slightly less than the two counts added together.

369,244 reviews are clean. every row has a real customer_id, real review text,
and is not a ghost order.

In [0]:
try:
    existing_gold = spark.table("bharatmart.ml.gld_review_insights")
    scored_ids = existing_gold.select("review_id")
    new_reviews_to_score = text_reviews_df.join(scored_ids, on="review_id", how="left_anti")

    print(f"total valid reviews : {text_reviews_df.count():,}")
    print(f"already in gold : {scored_ids.count():,}")
    print(f"new reviews to score : {new_reviews_to_score.count():,}")

except:
    # gold table does not exist yet, this is the first run
    new_reviews_to_score = text_reviews_df
    existing_gold  = None
    print("first run, no gold table found, scoring all reviews")

369,230 already scored in the Gold table from the previous run.

105 new reviews arrived in Silver since then. the anti-join found them by
comparing every review_id in Silver against every review_id in Gold and
returning only the ones with no match.

ai_query() will run on 105 rows tonight instead of 369,244.
the full dataset still gets updated churn and RFM scores in the join step
because customer risk changes every night even if the review text does not.

**Test Prompt on 5 Rows**

In [0]:
sample_df = new_reviews_to_score.limit(5)

test_result = sample_df.withColumn(
    "sentiment_raw",
    expr("""
        ai_query(
            'databricks-llama-4-maverick',
            concat(
                'You are a product review analyser. Read the review below and respond with ONLY valid JSON, no explanation, no markdown. ',
                'Use this exact format: {"sentiment": "positive", "score": 0.85, "issues": []} ',
                'Allowed sentiment values: positive, neutral, negative. ',
                'Score is your confidence 0.0 to 1.0. ',
                'Issues is an array of short phrases describing specific problems mentioned (empty array if none). ',
                'Review: ', review_text
            )
        )
    """)
)

print("test run done")

In [0]:
display(test_result.select("review_id", "rating", "review_text", "sentiment_raw"))

all 5 test rows returned valid JSON. the prompt is working correctly.

sentiment is accurate. rating 5 reviews came back positive, rating 3 came back neutral,
rating 2 came back negative. the model is reading the text, not just the star rating.

confidence scores look sensible. 0.95 for "Amazing quality" and 0.9 for "Terrible experience"
are clear signals. 0.7 for "Okay purchase. Meets basic expectations." is lower, which is
correct since neutral reviews are genuinely ambiguous.

issues array is working. REV0000000004 returned ["unhelpful customer support"] from the text
"Customer support was unhelpful too." that is exactly the extraction the pipeline is designed for.
the other four reviews had no complaints so issues came back as empty array.

the endpoint name changed from databricks-meta-llama-3-1-70b-instruct to
databricks-meta-llama-3-3-70b-instruct. the old endpoint was retired December 2024.
same fix is applied to the full batch cell below.

prompt is confirmed. safe to run on all 369,230 reviews.

**Verify JSON Parses Correctly**

In [0]:
test_parsed = test_result.select(
    col("review_id"),
    col("rating"),
    col("review_text"),
    get_json_object(col("sentiment_raw"), "$.sentiment").alias("sentiment"),
    get_json_object(col("sentiment_raw"), "$.score").cast("double").alias("score"),
    get_json_object(col("sentiment_raw"), "$.issues").alias("issues"),
)

display(test_parsed)

get_json_object() is extracting all three fields correctly.

sentiment, score and issues are clean separate columns now.
REV0000000004 shows the issues array working end to end,
"unhelpful customer support" extracted cleanly from the raw JSON string.

score is casting to double correctly, 0.95 and 0.7 not strings.

all 5 rows parsed. zero failures on the test sample.
the extraction logic is confirmed, safe to use on the full batch.

**Run ai_query on All Reviews**

In [0]:
scored_df = new_reviews_to_score.withColumn(
    "sentiment_raw",
    expr("""
        ai_query(
            'databricks-llama-4-maverick',
            concat(
                'You are a product review analyser. Read the review below and respond with ONLY valid JSON, no explanation, no markdown. ',
                'Use this exact format: {"sentiment": "positive", "score": 0.85, "issues": []} ',
                'Allowed sentiment values: positive, neutral, negative. ',
                'Score is your confidence 0.0 to 1.0. ',
                'Issues is an array of short phrases describing specific problems mentioned (empty array if none). ',
                'Review: ', review_text
            )
        )
    """)
)

scored_df.write.mode("overwrite").saveAsTable("bharatmart.ml.scored_temp")
scored_df = spark.table("bharatmart.ml.scored_temp")
print(f"ai_query done : {scored_df.count():,} reviews scored")

369,230 reviews scored successfully in first run after that it update according to new data comes. zero dropped during ai_query().

results written to bharatmart.ml.scored_temp first, then reloaded.
this means the LLM calls ran exactly once. the temp table acts as a checkpoint,
so if any downstream cell fails the scored data is not lost.

took roughly 40 minutes on Serverless Free Edition due to rate limiting on ai_query().
in a paid workspace with higher throughput limits this would be significantly faster.

**Extract JSON Fields**

In [0]:
parsed_df = scored_df.select(
    col("review_id"),
    col("customer_id"),
    col("product_id"),
    col("order_id"),
    col("rating"),
    col("review_text"),
    col("review_date"),
    get_json_object(col("sentiment_raw"), "$.sentiment").alias("sentiment"),
    get_json_object(col("sentiment_raw"), "$.score").cast("double").alias("sentiment_score"),
    get_json_object(col("sentiment_raw"), "$.issues").alias("issues"),
    col("sentiment_raw"),
)

display(parsed_df.limit(10))

all three fields extracted cleanly. sentiment, sentiment_score and issues
are separate columns, sentiment_raw kept alongside for debugging parse failures.

one row worth noting. REV0000000009 is rated 3 stars with text
"Decent product. Delivery was a bit slow though." the model returned
neutral sentiment with issues: ["slow delivery"].

that is the issues array doing its job. a 3 star review looks average in
the rating column but the model pulled out a specific operational complaint.
the retention team can filter on issues and find "slow delivery" complaints
across thousands of reviews, regardless of what star rating the customer gave.

zero parse failures in this sample. all sentiment_raw values returned
valid JSON that get_json_object() could extract from.

**Check Parse Failures**

In [0]:
failed    = parsed_df.filter(col("sentiment").isNull()).count()
succeeded = parsed_df.filter(col("sentiment").isNotNull()).count()

print(f"parsed ok : {succeeded:,}")
print(f"parse failed : {failed:,}  (null sentiment, raw response kept for inspection)")

zero parse failures across all 369,230 reviews.

the model-agnostic JSON prompt from Roumeliotis et al. (2024) held up
on the full dataset. every single ai_query() call returned valid JSON
that get_json_object() could extract from.

the sentiment_raw column is kept in parsed_df anyway. if a future run
with different data produces failures, the raw response is there to
inspect without rerunning ai_query().

**Add sentiment_label Column**

In [0]:
labeled_df = parsed_df \
    .withColumn(
        "sentiment_label",
        when(col("sentiment") == "positive",  1)
        .when(col("sentiment") == "neutral",  0)
        .when(col("sentiment") == "negative", -1)
        .otherwise(None)  # parse failures get null, not dropped
    ) \
    .withColumn("scored_date", current_date())

print("sentiment_label added")

In [0]:
display(labeled_df.select(
    "review_id", "customer_id", "rating",
    "sentiment", "sentiment_score", "sentiment_label", "issues"
).limit(5))

sentiment_label is mapping correctly. positive to 1, neutral to 0, negative to -1.

REV0000000003 is the most useful row here. rating 3, neutral sentiment, label 0.
the numeric label makes aggregation straightforward — avg(sentiment_label) per customer
gives a single number that churn features can consume directly.

scored_date will be added as today's date on every run, so the dashboard
can track how sentiment shifts over time as new reviews come in each night.

**Sentiment Distribution**

In [0]:
dist = labeled_df.groupBy("sentiment").count().orderBy(desc("count"))
display(dist)

positive reviews dominate at 275,596 (74.6% of scored reviews).
negative is 67,381 (18.2%) and neutral is 26,253 (7.1%).

this is a healthy distribution for an e-commerce platform.
roughly 1 in 5 reviews is negative, which is enough signal to act on
but not so high that the platform has a systemic quality problem.

the 67,381 negative reviews are the most valuable rows in this table.
each one has an issues array with specific complaints. that is what
the dashboard and Genie Space will surface for the retention team.

total adds to 369,230, confirming zero parse failures from Cell 10.

**Rating vs Sentiment Alignment Check**

In [0]:
hidden_dissatisfied = labeled_df.filter(
    (col("rating") >= 4) & (col("sentiment") == "negative")
)

total = labeled_df.count()
hidden = hidden_dissatisfied.count()

if total > 0:
    print(f"4-5 star reviews with negative sentiment : {hidden:,}  ({hidden/total*100:.1f}%)")
else:
    print("no new reviews scored this run — skipping hidden dissatisfied calc")

In [0]:
display(hidden_dissatisfied.select(
    "review_id", "customer_id", "rating", "sentiment", "issues", "review_text"
).limit(5))

40,970 reviews gave 4 or 5 stars but the text is negative. that is 11.1% of all scored reviews.

these are customers who clicked a high star rating but wrote a complaint.
this is exactly the hidden dissatisfaction signal the paper describes,
patterns that star ratings alone completely miss.

the sample rows show what this looks like in practice. REV10000057 gave 5 stars
but wrote "the thing shown is completely different from what we get" with issues
["product mismatch", "false advertising"]. the rating says satisfied, the text says the opposite.

the two most common issues in this sample are product mismatch and unsatisfactory experience.
these are operational problems the business can actually fix, not vague complaints.

40,970 customers in this group are a retention risk that would be invisible
without text analysis. they are not flagged by rating alone.

**Cross Reference With Churn Predictions**

In [0]:
churn_df = spark.table("bharatmart.ml.gld_churn_predictions").select(
    "customer_id", "churn_probability", "churn_risk_band", "top_churn_reason"
)

rfm_df = spark.table("bharatmart.ml.gld_rfm_segments").select(
    "customer_id", "rfm_segment", "retention_priority"
)

print(f"churn predictions loaded : {churn_df.count():,} customers")
print(f"rfm segments loaded : {rfm_df.count():,} customers")

both tables loaded with matching row counts. 94,175 customers in both
churn predictions and RFM segments, which is expected since both models
scored the same customer base in the first run after this it will update every time according to new data comes.

the join in the next cell will be clean with no count mismatch between
the two tables.

**Join to Churn and RFM**

In [0]:
# 1. Combine Old LLM Scores with New LLM Scores
if existing_gold is not None:
    # Select only the static review columns from the old gold table
    old_reviews_llm_only = existing_gold.select(
        "review_id", "customer_id", "product_id", "order_id", "rating", 
        "review_text", "review_date", "sentiment", "sentiment_score", 
        "sentiment_label", "issues", "scored_date"
    )
    
    # Make sure labeled_df has the exact same columns
    new_reviews_llm_only = labeled_df.select(
        "review_id", "customer_id", "product_id", "order_id", "rating", 
        "review_text", "review_date", "sentiment", "sentiment_score", 
        "sentiment_label", "issues", "scored_date"
    )
    
    # Union them together
    all_scored_reviews = old_reviews_llm_only.unionByName(new_reviews_llm_only)
    print(f"combined old and new scores : {all_scored_reviews.count():,}")
else:
    all_scored_reviews = labeled_df

# 2. Join ALL reviews to tonight's fresh churn and RFM tables
insights_joined = all_scored_reviews \
    .join(churn_df, on="customer_id", how="left") \
    .join(rfm_df,   on="customer_id", how="left")

print(f"reviews after join : {insights_joined.count():,}")

In [0]:
display(insights_joined.select(
    "review_id", "customer_id", "rfm_segment", "retention_priority",
    "churn_probability", "rating", "sentiment", "issues"
).filter(col("retention_priority") == "Urgent").limit(5))

369,230 rows after join, same as before. no reviews lost, the left join
preserved all scored reviews including customers not in the churn table.

the Urgent filter shows what this join is designed to find. these are
Champions customers, the highest value segment from Model 2, who the
churn model flagged with probability above 0.5.

REV0000001130 is the most critical row in this sample. CUST0058596 is a
Champions customer with churn probability 0.59, rated 2 stars, sentiment
negative, issues ["late delivery", "damaged product"]. this customer is
high value, close to leaving, and actively complaining about two specific
operational problems.

a generic retention voucher will not fix late delivery or damaged packaging.
the issues array is what makes a targeted response possible.

**Urgent Customers With Negative Reviews**

In [0]:
urgent_negative = insights_joined.filter(
    (col("retention_priority") == "Urgent") &
    (col("sentiment") == "negative")
)

print(f"urgent + negative : {urgent_negative.count():,}")

120 customers are Champions, flagged as Urgent by the churn model,
and have left a negative review.

from the 233 Urgent customers identified in Model 2, 120 of them have
also written a negative review. that is more than half the Urgent group
actively complaining in text.

these 120 customers are the single highest priority group in the entire
BharatMart customer base. highest value segment, closest to leaving,
and telling us exactly why through the issues array.

this is the cross-model output that neither Model 1 nor Model 2 could
produce alone. churn probability identifies the risk. RFM segment
identifies the value. sentiment analysis identifies the reason.
all three together in one row.

**Write gld_review_insights**

In [0]:
final_df = insights_joined.select(
    "review_id",
    "customer_id",
    "product_id",
    "order_id",
    "rating",
    "review_text",
    "review_date",
    "sentiment",
    "sentiment_score",
    "sentiment_label",
    "issues",
    "churn_probability",
    "churn_risk_band",
    "top_churn_reason",
    "retention_priority",
    "rfm_segment",
    "scored_date",
)

(
    final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bharatmart.ml.gld_review_insights")
)

print("wrote to bharatmart.ml.gld_review_insights")

369,230 rows written to bharatmart.ml.gld_review_insights.

overwriteSchema=true means future retraining runs with schema changes
will not fail the write. same pattern used across all ML output tables
in this pipeline.

the table now has sentiment, churn probability, RFM segment and
retention priority in one row per review. the AIBI dashboard and
Genie Space read directly from this table.

**Verify the Table**

In [0]:
result = spark.table("bharatmart.ml.gld_review_insights")

print(f"gld_review_insights : {result.count():,} rows")

In [0]:
summary = result.groupBy("sentiment").agg(
    count("*").alias("count"),
    spark_round(avg("sentiment_score"), 3).alias("avg_score")
).orderBy(desc("count"))

display(summary)

369,230 rows confirmed in the table. write was clean.

sentiment counts match Cell 12 exactly. positive 275,596, negative 67,381,
neutral 26,253. nothing dropped or duplicated during the write.

the avg_score column shows something interesting. positive and negative
reviews both score around 0.92 confidence, meaning the model is very
certain when it sees clear positive or negative text.

neutral reviews score 0.688, much lower. this makes sense. neutral text
like "okay purchase, meets basic expectations" is genuinely ambiguous
and the model is correctly expressing lower confidence on it.

lower confidence on neutral is not a problem. it is the model being
honest about ambiguous input, which is exactly what the score field
is designed to communicate.

**Sentiment by Churn Risk Band**

In [0]:
by_risk = result.groupBy("churn_risk_band", "sentiment") \
    .count() \
    .orderBy("churn_risk_band", "sentiment")

display(by_risk)

negative review rate is almost identical across all four churn risk bands.
Critical 17.2%, High 18.1%, Medium 18.5%, Low 18.3%.

this is a surprising finding. customers at high churn risk are not leaving
significantly more negative reviews than low risk customers. complaints are
spread evenly across the customer base regardless of churn risk.

this tells us two things. first, negative reviews alone do not predict churn.
a customer can be unhappy in text but still keep buying. second, the combination
of churn risk plus negative sentiment is what matters, not either signal alone.

that is exactly what the 120 Urgent negative customers from Cell 16 represent.
high churn risk AND negative sentiment together is the signal worth acting on.
not negative sentiment by itself.

2 reviews have null churn_risk_band. these are customers in the reviews table
who are not in gld_churn_predictions, likely customers added after the last
churn scoring run.

**Final Summary**

In [0]:
total_scored = result.count()
positive_count = result.filter(col("sentiment") == "positive").count()
neutral_count = result.filter(col("sentiment") == "neutral").count()
negative_count = result.filter(col("sentiment") == "negative").count()
hidden = result.filter((col("rating") >= 4) & (col("sentiment") == "negative")).count()
urgent_neg = result.filter((col("retention_priority") == "Urgent") & (col("sentiment") == "negative")).count()
parse_failed = result.filter(col("sentiment").isNull()).count()

if total_scored > 0:
    print(f"positive : {positive_count:,}  ({positive_count/total_scored*100:.1f}%)")
    print(f"neutral : {neutral_count:,}  ({neutral_count/total_scored*100:.1f}%)")
    print(f"negative : {negative_count:,}  ({negative_count/total_scored*100:.1f}%)")
    print(f"hidden dissatisfied : {hidden:,}  ({hidden/total_scored*100:.1f}%)")
else:
    print("no new reviews scored this run")

model 5 is complete.

369,230 reviews scored with zero parse failures. the model-agnostic JSON
prompt held up across the entire dataset without a single malformed response.

the headline number is 120. these are Champions customers who are Urgent
churn risks and have left a negative review with specific complaints in
the issues array. this group is invisible without all three models working
together. churn probability alone finds the risk. RFM segment finds the
value. sentiment finds the reason.

40,970 hidden dissatisfied customers gave 4 or 5 stars but wrote negative
text. these would be completely missed by any rating-based analysis.

gld_review_insights is ready for the AIBI dashboard and Genie Space.
the retention team can now query: which Urgent customers complained about
late delivery this week, filter by issues array, and action directly.

In [0]:
%skip
# First, look at the history to find the version number before your mistake
display(spark.sql("DESCRIBE HISTORY bharatmart.ml.gld_review_insights"))

# Find the version number where the table had 369,000+ rows, then run:
# spark.sql("RESTORE TABLE bharatmart.ml.gld_review_insights TO VERSION AS OF <that_version_number>")

In [0]:
%skip
%sql
RESTORE TABLE bharatmart.ml.gld_review_insights TO VERSION AS OF 1